# 002 - Atividade Pratica - Lakehouse - Extracao


## Extração

Lê todas as collections do MongoDB Atlas e grava cada collection como JSON no volume `workspace.landing.dados`.

A connection string não deve ser versionada no GitHub. Informe o valor no widget `mongodb_uri` ao executar o notebook no Databricks.


In [ ]:
%pip install pymongo certifi


In [ ]:
dbutils.widgets.text("mongodb_uri", "", "MongoDB Atlas URI")
dbutils.widgets.text("mongodb_database", "ai_job_market", "MongoDB Database")

mongodb_uri = dbutils.widgets.get("mongodb_uri")
mongodb_database = dbutils.widgets.get("mongodb_database")

if not mongodb_uri:
    raise ValueError("Informe a connection string do MongoDB Atlas no widget 'mongodb_uri'.")


In [ ]:
import json
import certifi
from pymongo import MongoClient

caminho_landing = "/Volumes/workspace/landing/dados"

collections = {
    "job_title": "Job_Title",
    "industry": "Industry",
    "company_size": "Company_Size",
    "location": "Location",
    "ai_adoption_level": "AI_Adoption_Level",
    "automation_risk": "Automation_Risk",
    "required_skills": "Required_Skills",
    "salary_usd": "Salary_USD",
    "remote_friendly": "Remote_Friendly",
    "job_growth_projection": "Job_Growth_Projection",
}


In [ ]:
dbutils.fs.mkdirs(caminho_landing)

for arquivo in dbutils.fs.ls(caminho_landing):
    if arquivo.name.endswith(".json"):
        dbutils.fs.rm(arquivo.path)


In [ ]:
client = MongoClient(
    mongodb_uri,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=20000,
)
database = client[mongodb_database]

for collection_name, source_column in collections.items():
    docs = database[collection_name].find({}, {"_id": 0}).sort("id_linha", 1)
    linhas_json = []

    for doc in docs:
        value = doc.get("valor", doc.get(source_column))
        linhas_json.append(
            json.dumps(
                {
                    "id_linha": doc.get("id_linha"),
                    source_column: value,
                    "collection_origem": collection_name,
                },
                ensure_ascii=False,
            )
        )

    dbutils.fs.put(
        f"{caminho_landing}/{collection_name}.json",
        "\n".join(linhas_json) + "\n",
        overwrite=True,
    )

client.close()


In [ ]:
display(dbutils.fs.ls(caminho_landing))
